## Configuration ##

In [1]:
import os
import json
import pathlib
                "import requests",


                "def _table(db, internal_name):",
                "    return next(t for t in db.tables if t.internal_name == internal_name)",


                "def _column(table, internal_name):",
                "    return next(c for c in table.columns if c.internal_name == internal_name)",


                "def create_view_payload_with_aliases(db):",
                "    casualty = _table(db, \"casualty\")",
                "    collision = _table(db, \"collision\")",
                "    vehicle = _table(db, \"vehicle\")",
                "",
                "    # Assign stable aliases to datasources to avoid duplicate table names in generated SQL",
                "    datasources = [",
                "        {\"id\": casualty.id, \"alias\": \"cas\"},",
                "        {\"id\": collision.id, \"alias\": \"col\"},",
                "        {\"id\": vehicle.id, \"alias\": \"veh\"},",
                "    ]",
                "",
                "    def col_entry(table, internal_name, alias):",
                "        return {\"id\": _column(table, internal_name).id, \"datasource_alias\": alias}",
                "",
                "    columns = [",
                "        col_entry(casualty, \"casualty_id\", \"cas\"),",
                "        col_entry(collision, \"collision_index\", \"col\"),",
                "        col_entry(casualty, \"casualty_severity\", \"cas\"),",
                "        col_entry(collision, \"road_type\", \"col\"),",
                "        col_entry(collision, \"speed_limit\", \"col\"),",
                "        col_entry(collision, \"weather_conditions\", \"col\"),",
                "        col_entry(collision, \"light_conditions\", \"col\"),",
                "        col_entry(collision, \"road_surface_conditions\", \"col\"),",
                "        col_entry(collision, \"time\", \"col\"),",
                "        col_entry(collision, \"day_of_week\", \"col\"),",
                "        col_entry(collision, \"number_of_vehicles\", \"col\"),",
                "        col_entry(vehicle, \"vehicle_type\", \"veh\"),",
                "    ]",
                "",
                "    joins = [",
                "        {",
                "            \"type\": \"inner\",",
                "            \"datasource_alias\": \"col\",",
                "            \"conditionals\": [",
                "                {\"column_id\": _column(casualty, \"collision_index\").id, \"foreign_column_id\": _column(collision, \"collision_index\").id}],",
                "        },",
                "        {",
                "            \"type\": \"inner\",",
                "            \"datasource_alias\": \"veh\",",
                "            \"conditionals\": [",
                "                {\"column_id\": _column(casualty, \"collision_index\").id, \"foreign_column_id\": _column(vehicle, \"collision_index\").id},",
                "                {\"column_id\": _column(casualty, \"vehicle_reference\").id, \"foreign_column_id\": _column(vehicle, \"vehicle_reference\").id},",
                "            ],",
                "        },",
                "    ]",
                "",
                "    payload = {",
                "        \"name\": \"v_ml_features\",",
                "        \"query\": {",
                "            \"datasources\": datasources,",
                "            \"columns\": columns,",
                "            \"joins\": joins,",
                "            \"filters\": None,",
                "            \"orders\": None,",
                "        },",
                "        \"is_public\": True,",
                "        \"is_schema_public\": True,",
                "    }",
                "    return payload",
                "",
                "",
                "db = client.get_database(DB_ID)",
                "payload = create_view_payload_with_aliases(db)",
                "",
                "# Try using the client helper first (may raise if mapper fails); fall back to direct POST",
                "try:",
                "    print(\"Attempting create_view via RestClient.create_view()\")",
                "    result = client.create_view(database_id=DB_ID, name=payload['name'], query=payload['query'], is_public=payload['is_public'], is_schema_public=payload['is_schema_public'])",
                "    # result may be a DTO or a dict depending on client version",
                "    created_id = getattr(result, 'id', None) or (result.get('id') if isinstance(result, dict) else None)",
                "    print(f\"Created view (via client): {created_id or result}\")",
                "except Exception as e:",
                "    print(\"RestClient.create_view failed, falling back to direct HTTP POST (this prints exception)\")",
                "    print(e)",
                "    response = requests.post(",
                "        f\"{HOST}/api/v1/database/{DB_ID}/view\",",
                "        auth=auth,",
                "        headers={\"Content-Type\": \"application/json\", \"Accept\": \"application/json\"},",
                "        json=payload,",
                "        verify=True,",
                "    )",
                "    if response.status_code in (200, 201):",
                "        created = response.json()",
                "        print(f\"Created view: {created.get('id') or created.get('view_id') or created}\")",
                "    else:",
                "        print(f\"Failed to create view: {response.status_code} {response.text}\")",
                "        print(\"If you see 403: the executing account is not the database owner.\")",
                "        print(\"If you see 417 and a SQL error about duplicate aliases: I can further adjust datasource ordering/aliases.\")"
        col("first_road_class",                             "varchar", size=255),
        col("first_road_number",                            "double"),
        col("road_type",                                    "varchar", size=255),
        col("speed_limit",                                  "double"),
        col("junction_detail",                              "varchar", size=255),
        col("junction_control",                             "varchar", size=255),
        col("second_road_class",                            "varchar", size=255),
        col("second_road_number",                           "double"),
        col("pedestrian_crossing",                          "varchar", size=255),
        col("light_conditions",                             "varchar", size=255),
        col("weather_conditions",                           "varchar", size=255),
        col("road_surface_conditions",                      "varchar", size=255),
        col("special_conditions_at_site",                   "varchar", size=255),
        col("carriageway_hazards",                          "varchar", size=255),
        col("urban_or_rural_area",                          "varchar", size=255),
        col("did_police_officer_attend_scene_of_accident",  "double"),
        col("trunk_road_flag",                              "double"),
        col("lsoa_of_accident_location",                    "varchar", size=255),
        col("enhanced_severity_collision",                  "varchar", size=255),
        col("collision_injury_based",                       "double"),
        col("collision_adjusted_severity_serious",          "double"),
        col("collision_adjusted_severity_slight",           "double"),
    ]
)

vehicle_json = build_table_json(
    name="vehicle",
    description="One row per vehicle involved in a recorded collision.",
    primary_key_col="vehicle_id",
    columns=[
        col("vehicle_id",                       "double"),
        col("collision_index",                  "varchar", size=255),
        col("vehicle_reference",                "double"),
        col("vehicle_type",                     "varchar", size=255),
        col("towing_and_articulation",          "varchar", size=255),
        col("vehicle_manoeuvre",                "varchar", size=255),
        col("vehicle_direction_from",           "varchar", size=255),
        col("vehicle_direction_to",             "varchar", size=255),
        col("vehicle_location_restricted_lane", "varchar", size=255),
        col("junction_location",                "varchar", size=255),
        col("skidding_and_overturning",         "varchar", size=255),
        col("hit_object_in_carriageway",        "varchar", size=255),
        col("vehicle_leaving_carriageway",      "varchar", size=255),
        col("hit_object_off_carriageway",       "varchar", size=255),
        col("first_point_of_impact",            "varchar", size=255),
        col("vehicle_left_hand_drive",          "double"),
        col("journey_purpose_of_driver",        "varchar", size=255),
        col("sex_of_driver",                    "varchar", size=50),
        col("age_of_driver",                    "double"),
        col("age_band_of_driver",               "varchar", size=50),
        col("engine_capacity_cc",               "double"),
        col("propulsion_code",                  "varchar", size=255),
        col("age_of_vehicle",                   "double"),
        col("generic_make_model",               "varchar", size=255),
        col("driver_imd_decile",                "double"),
        col("lsoa_of_driver",                   "varchar", size=255),
        col("escooter_flag",                    "double"),
        col("driver_distance_banding",          "varchar", size=255),
    ]
)

casualty_json = build_table_json(
    name="casualty",
    description="One row per casualty in a recorded collision.",
    primary_key_col="casualty_id",
    columns=[
        col("casualty_id",                          "double"),
        col("collision_index",                      "varchar", size=255),
        col("vehicle_reference",                    "double"),
        col("casualty_reference",                   "double"),
        col("casualty_class",                       "varchar", size=255),
        col("sex_of_casualty",                      "varchar", size=50),
        col("age_of_casualty",                      "double"),
        col("age_band_of_casualty",                 "varchar", size=50),
        col("casualty_severity",                    "varchar", size=255),
        col("pedestrian_location",                  "varchar", size=255),
        col("pedestrian_movement",                  "varchar", size=255),
        col("car_passenger",                        "varchar", size=255),
        col("bus_or_coach_passenger",               "varchar", size=255),
        col("pedestrian_road_maintenance_worker",   "varchar", size=255),
        col("casualty_type",                        "varchar", size=255),
        col("casualty_imd_decile",                  "double"),
        col("lsoa_of_casualty",                     "varchar", size=255),
        col("enhanced_casualty_severity",           "varchar", size=255),
        col("casualty_injury_based",                "double"),
        col("casualty_adjusted_severity_serious",   "double"),
        col("casualty_adjusted_severity_slight",    "double"),
        col("casualty_distance_banding",            "varchar", size=255),
    ]
)

# Send them
import requests

for table_json in [collision_json, vehicle_json, casualty_json]:
    url = f"{HOST}/api/v1/database/{DB_ID}/table"
    response = requests.post(
        url,
        auth=(USERNAME, PASSWORD),
        headers={"Content-Type": "application/json", "Accept": "application/json"},
        json=table_json,
        verify=True
    )
    if response.status_code in (200, 201):
        print(f"{table_json['name']} (id={response.json().get('id')})")
    else:
        print(f"{table_json['name']}: {response.status_code} {response.text}")

Configuration loaded.


from dbrepo.RestClient import RestClient
url = f"{API_BASE}/database/{DB_ID}"

client =RestClient(endpoint=HOST, username=USERNAME, password=PASSWORD, secure=True)


for t in tables:
    result = client.create_table(
        database_id=DB_ID,
        name=t['name'],
        is_public=t["is_public"],
        is_schema_public=t["is_schema_public"],
        dataframe=t["dataframe"],
        description=t["description"],
        with_data=False
    )
    print
    print(f"done (table_id={result.id})")

## Create citable identifier ##

In [3]:
import uuid
#This should contain license OGL-UK-3.0 instead of CC-BY-4.0, but DBRepo API wont allow it
payload = {
  "type": "database",
  "titles": [
    {
        "title": "UK Road Safety Open Data 2023",
        "language": "en",
        "type": "Subtitle"
    }
  ],
    "descriptions": [
        {
            "description": (
                "Road safety and traffic collision data for Great Britain for the year 2023, "
                "originally published by the UK Department for Transport under the "
                "Open Government Licence v3.0. "
                "Covers reported accidents, involved vehicles, casualties, "
                "and associated road and environmental conditions. "
                "This relational database was created for academic purposes "
                "as part of the Data Stewardship course at TU Wien (Group 6, 2026)."
            ),
            "language": "en",
            "type": "Abstract"
        }
    ],
    "funders": [
        {
          "funder_name": "Department for Transport, United Kingdom"
        }
    ],
    "licenses": [
        {
            "identifier": "CC-BY-4.0", 
            "uri": "https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/",
            "description": "Open Government Licence v3.0"
        }
    ],
    "publisher": "Crown Copyright – Department for Transport, United Kingdom",
    "language": "en",
    "creators": [
        {
          "affiliation": "Department for Transport, United Kingdom",
          "creator_name": "Department for Transport, United Kingdom",
          "name_type": "Organizational",
          "affiliation_identifier": "https://www.gov.uk/government/organisations/department-for-transport"
        }
      ],
    
      "database_id": DB_ID,
      "publication_year": 2023,
      "related_identifiers": [
        {
            "value": "https://www.gov.uk/government/statistical-data-sets/road-safety-open-data",
            "type": "URL",
            "relation": "IsDerivedFrom"
        }
    ],
}

r = requests.post(
    f"{HOST}/api/v1/identifier",
    auth=(USERNAME, PASSWORD),
    headers={"Content-Type": "application/json", "Accept": "application/json"},
    json=payload,
    verify=True
)
print(r.status_code, r.text)

201 {"id":"b7777a35-075c-4320-a38a-d624b933e4f5","links":{"self":"/api/v1/identifier/b7777a35-075c-4320-a38a-d624b933e4f5","data":null,"self_html":"/pid/b7777a35-075c-4320-a38a-d624b933e4f5","dashboard_html":"/d/cfmr0a560xo1sb"},"type":"database","titles":[{"id":"23621b7a-a413-404c-9037-9fdcbfde3c72","title":"UK Road Safety Open Data 2023","language":"en","type":"Subtitle"}],"descriptions":[{"id":"52ca4df5-5200-4f9c-978a-9972de8bdb67","description":"Road safety and traffic collision data for Great Britain for the year 2023, originally published by the UK Department for Transport under the Open Government Licence v3.0. Covers reported accidents, involved vehicles, casualties, and associated road and environmental conditions. This relational database was created for academic purposes as part of the Data Stewardship course at TU Wien (Group 6, 2026).","language":"en","type":"Abstract"}],"funders":[{"id":"0bc8e6c6-333c-4936-83bf-3b0567943c9f","funder_name":"Department for Transport, United

## Give other members access ##

In [4]:


url = f"{HOST}/api/v1/database/{DB_ID}/access/12549571"

payload = {"type" : "write_all"}

r = requests.post(url, auth=auth, json=payload)




print(r.status_code, r.text)

403 {"status":"FORBIDDEN","message":"Failed to create access to user 12549571: already has access","code":"error.request.forbidden"}


In [5]:
url = f"{HOST}/api/v1/database/{DB_ID}/access/12030168"
payload = {"type" : "write_all"}

r = requests.post(url, auth=auth, json=payload)

print(r.status_code, r.text)

202 


In [ ]:
url = f"{HOST}/api/v1/database/{DB_ID}/access/12226609"
payload = {"type" : "write_all"}

r = requests.post(url, auth=auth, json=payload)

print(r.status_code, r.text)

## VIEW

In [ ]:
from dbrepo.RestClient import RestClient
from dbrepo.api.dto import QueryDefinition, JoinDefinition, JoinType, ConditionalDefinition

# Credentials for the DBRepo creation step
USERNAME = "12549571"
PASSWORD = ""
auth = (USERNAME, PASSWORD)

client = RestClient(endpoint=HOST, username=USERNAME, password=PASSWORD, secure=True)
print("DBRepo client ready for view creation.")

DBRepo client ready for view creation.


In [5]:
import requests


def _table(db, internal_name):
    return next(t for t in db.tables if t.internal_name == internal_name)


def _column(table, internal_name):
    return next(c for c in table.columns if c.internal_name == internal_name)


def create_view_payload(db):
    casualty = _table(db, "casualty")
    collision = _table(db, "collision")
    vehicle = _table(db, "vehicle")

    return {
        "name": "v_ml_features",
        "query": {
            "columns": [
                {"id": _column(casualty, "casualty_id").id},
                {"id": _column(collision, "collision_index").id},
                {"id": _column(casualty, "casualty_severity").id},
                {"id": _column(collision, "road_type").id},
                {"id": _column(collision, "speed_limit").id},
                {"id": _column(collision, "weather_conditions").id},
                {"id": _column(collision, "light_conditions").id},
                {"id": _column(collision, "road_surface_conditions").id},
                {"id": _column(collision, "time").id},
                {"id": _column(collision, "day_of_week").id},
                {"id": _column(collision, "number_of_vehicles").id},
                {"id": _column(vehicle, "vehicle_type").id},
            ],
            "datasource_ids": [casualty.id, collision.id, vehicle.id],
            "joins": [
                {
                    "type": "inner",
                    "datasource_id": collision.id,
                    "conditionals": [
                        {
                            "column_id": _column(casualty, "collision_index").id,
                            "foreign_column_id": _column(collision, "collision_index").id,
                        }
                    ],
                },
                {
                    "type": "inner",
                    "datasource_id": vehicle.id,
                    "conditionals": [
                        {
                            "column_id": _column(casualty, "collision_index").id,
                            "foreign_column_id": _column(vehicle, "collision_index").id,
                        },
                        {
                            "column_id": _column(casualty, "vehicle_reference").id,
                            "foreign_column_id": _column(vehicle, "vehicle_reference").id,
                        },
                    ],
                },
            ],
            "filters": None,
            "orders": None,
        },
        "is_public": True,
        "is_schema_public": True,
    }


db = client.get_database(DB_ID)
payload = create_view_payload(db)
response = requests.post(
    f"{HOST}/api/v1/database/{DB_ID}/view",
    auth=auth,
    headers={"Content-Type": "application/json", "Accept": "application/json"},
    json=payload,
    verify=True,
)
if response.status_code in (200, 201):
    created = response.json()
    print(f"Created view: {created.get('id') or created.get('view_id') or created}")
else:
    print(f"Failed to create view: {response.status_code} {response.text}")

Failed to create view: 403 {"status":"FORBIDDEN","message":"Failed to create view: not the database owner","code":"error.request.forbidden"}
